In [19]:
import os
print("Current Folder:", os.getcwd())
print("\nFiles in this folder:", os.listdir('.'))

try:
    print("\nFiles in the folder above this one (../):", os.listdir('..'))
except Exception as e:
    print("\nCannot access parent folder:", e)

Current Folder: /Users/nishi/Downloads/Compass EV Project/Compass_EVProject/notebooks/recommender

Files in this folder: ['03_neural_calibration_training.ipynb', '01_datapreprocessing.ipynb', '06_price_pytorch_model.ipynb', '05_price_baseline_model.ipynb', '04_model_export.ipynb', '02_weighted_scoring_simulation.ipynb']

Files in the folder above this one (../): ['recommender']


In [20]:
import os
print("Current Folder:", os.getcwd())
print("\nFiles in this folder:", os.listdir('.'))

try:
    print("\nFiles in the folder above this one (../):", os.listdir('..'))
except Exception as e:
    print("\nCannot access parent folder:", e)

Current Folder: /Users/nishi/Downloads/Compass EV Project/Compass_EVProject/notebooks/recommender

Files in this folder: ['03_neural_calibration_training.ipynb', '01_datapreprocessing.ipynb', '06_price_pytorch_model.ipynb', '05_price_baseline_model.ipynb', '04_model_export.ipynb', '02_weighted_scoring_simulation.ipynb']

Files in the folder above this one (../): ['recommender']


In [21]:
import pandas as pd
import numpy as np
import os

file_name = 'compiled_vehicle_models_data.csv'

search_folders = [
    '.', '..', '../..',
    '/Users/nishi/Downloads',
    '/Users/nishi/Desktop',
    '/Users/nishi/Documents'
]

found_path = None
for folder in search_folders:
    potential_path = os.path.join(folder, file_name)
    if os.path.exists(potential_path):
        found_path = potential_path
        break

if found_path is None:
    print("❌ Critical Error: File missing.")
else:
    print(f"✅ Found the file hiding at: {found_path}")
    
    df = pd.read_csv(found_path)
    df.columns = [c.lower() for c in df.columns]
    
    if 'range_miles' not in df.columns:
        df['range_miles'] = df['range_km'] * 0.621371
        
    df = df.rename(columns={'model_name': 'Model', 'brand': 'Manufacturer', 'price_usd': 'Price_USD'})
    
    # Only use mainstream brands, but drop the price cap so we actually get cars
    mainstream_brands = ['Chevrolet', 'Nissan', 'Hyundai', 'Kia', 'Volkswagen', 'Ford', 'Toyota', 'Tesla']
    cleaned_df = df[df['Manufacturer'].isin(mainstream_brands)].copy()
    
    # Take the lowest available scrambled price for each model
    scoring_df = cleaned_df.groupby(['Manufacturer', 'Model']).agg({
        'Price_USD': 'min',
        'range_miles': 'max',
        'battery_capacity_kwh': 'max'
    }).reset_index().dropna()

    print(f"🧹 Data cleaned! We now have {len(scoring_df)} unique EV models.")

def recommend_evs(user_weights, data, top_n=5):
    df_scored = data.copy()
    
    def safe_scale(column, inverse=False):
        min_val, max_val = column.min(), column.max()
        if max_val == min_val: return 1.0 
        if inverse: return (max_val - column) / (max_val - min_val)
        return (column - min_val) / (max_val - min_val)

    df_scored['score_price'] = safe_scale(df_scored['Price_USD'], inverse=True)
    df_scored['score_range'] = safe_scale(df_scored['range_miles'])
    df_scored['score_battery'] = safe_scale(df_scored['battery_capacity_kwh'])
    
    df_scored['Final_Match_Score'] = (
        (df_scored['score_price'] * user_weights.get('price', 0)) +
        (df_scored['score_range'] * user_weights.get('range', 0)) +
        (df_scored['score_battery'] * user_weights.get('battery', 0))
    ) * 100
    
    return df_scored.sort_values(by='Final_Match_Score', ascending=False).head(top_n)

✅ Found the file hiding at: ../../compiled_vehicle_models_data.csv
🧹 Data cleaned! We now have 1 unique EV models.


In [22]:
# Define the user's priorities
budget_conscious_user = {
    'price': 0.60,   # 60% importance on keeping price low
    'range': 0.30,   # 30% importance on having high range
    'battery': 0.10  # 10% importance on battery size
}

print("🚗 Top Recommendations for the Budget-Conscious User:")
print("-" * 50)
results = recommend_evs(budget_conscious_user, scoring_df)

# Format the output beautifully
for index, row in results.iterrows():
    print(f"{row['Manufacturer']} {row['Model']}")
    print(f"  ↳ Match Score: {row['Final_Match_Score']:.1f}%")
    print(f"  ↳ Price: ${row['Price_USD']:,.0f} | Range: {row['range_miles']:.0f} miles\n")

🚗 Top Recommendations for the Budget-Conscious User:
--------------------------------------------------
Kia Niro EV
  ↳ Match Score: 100.0%
  ↳ Price: $34,769 | Range: 239 miles



In [23]:
# Define the user's priorities (must add up to 1.0)
# The ultimate band-aid: Only allow mainstream brands to bypass the corrupted luxury data
budget_brands = ['Chevrolet', 'Nissan', 'Hyundai', 'Kia', 'Volkswagen', 'Ford']
scoring_df = scoring_df[scoring_df['Manufacturer'].isin(budget_brands)]
budget_conscious_user = {
    'price': 0.60,   # 60% importance on keeping price low
    'range': 0.30,   # 30% importance on having high range
    'battery': 0.10  # 10% importance on battery size
}

print("🚗 Top Recommendations for the Budget-Conscious User:")
print("-" * 50)
results = recommend_evs(budget_conscious_user, scoring_df)

# Format the output beautifully
for index, row in results.iterrows():
    print(f"{row['Manufacturer']} {row['Model']}")
    print(f"  ↳ Match Score: {row['Final_Match_Score']:.1f}%")
    print(f"  ↳ Price: ${row['Price_USD']:,.0f} | Range: {row['range_miles']:.0f} miles")
    print()

🚗 Top Recommendations for the Budget-Conscious User:
--------------------------------------------------
Kia Niro EV
  ↳ Match Score: 100.0%
  ↳ Price: $34,769 | Range: 239 miles

